In [ ]:
import os
import requests
from tqdm import tqdm  # Pour une barre de progression de téléchargement

# URL de téléchargement de la base de données PASCAL VOC2012
data_url = "http://host.robots.ox.ac.uk/pascal/VOC/voc2012/VOCtrainval_11-May-2012.tar"

# Chemin local où télécharger et extraire les données
data_dir = "./pascal_voc2012"

# Créer le dossier si nécessaire
if not os.path.exists(data_dir):
    os.makedirs(data_dir)

# Chemin de destination pour enregistrer le fichier téléchargé
download_path = "VOCtrainval_11-May-2012.tar"
tar_filename = os.path.join(data_dir, "VOCtrainval_11-May-2012.tar")

# Vérifie si le fichier existe déjà pour éviter de le télécharger à nouveau
if not os.path.exists(tar_filename):
    print("Téléchargement en cours...")
    response = requests.get(data_url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024  # Taille des blocs de téléchargement en octets
    progress_bar = tqdm(total=total_size, unit='B', unit_scale=True)
    
    with open(tar_filename, 'wb') as file:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            file.write(data)
    progress_bar.close()
    print("Téléchargement terminé.")
else:
    print("Le fichier existe déjà.")

In [6]:
import os
import tarfile

# Chemin local où télécharger et extraire les données
data_dir = "./pascal_voc2012"

# Téléchargement du fichier tar contenant les données
tar_filename = os.path.join(data_dir, "VOCtrainval_11-May-2012.tar")

# Extraction du fichier tar
with tarfile.open(tar_filename, "r") as tar:
    tar.extractall(path=data_dir)

print("Données téléchargées et extraites avec succès.")


Données téléchargées et extraites avec succès.


In [7]:
import os
import shutil

# Chemin vers les dossiers d'images et de masques
image_dir = os.path.join(data_dir, "VOCdevkit", "VOC2012", "JPEGImages")
mask_dir = os.path.join(data_dir, "VOCdevkit", "VOC2012", "SegmentationClass")

# Chemin vers les dossiers de destination pour les images et les masques organisés
destination_image_dir = os.path.join(data_dir, "images")
destination_mask_dir = os.path.join(data_dir, "masks")

# Création des dossiers de destination
os.makedirs(destination_image_dir, exist_ok=True)
os.makedirs(destination_mask_dir, exist_ok=True)

# Copie des images vers le dossier de destination
for filename in os.listdir(image_dir):
    shutil.copy(os.path.join(image_dir, filename), os.path.join(destination_image_dir, filename))

# Copie des masques vers le dossier de destination
for filename in os.listdir(mask_dir):
    shutil.copy(os.path.join(mask_dir, filename), os.path.join(destination_mask_dir, filename))

print("Données organisées avec succès.")


Données organisées avec succès.


In [8]:
from PIL import Image
import numpy as np

# Chemins vers les dossiers d'images et de masques organisés
destination_image_dir = os.path.join(data_dir, "images")
destination_mask_dir = os.path.join(data_dir, "masks")

# Chemin vers le dossier de destination pour les images prétraitées et les masques
preprocessed_image_dir = os.path.join(data_dir, "preprocessed_images")
preprocessed_mask_dir = os.path.join(data_dir, "preprocessed_masks")

# Création des dossiers de destination pour les données prétraitées
os.makedirs(preprocessed_image_dir, exist_ok=True)
os.makedirs(preprocessed_mask_dir, exist_ok=True)

# Dimensions de redimensionnement souhaitées
target_size = (513, 513)

# Prétraitement des images et des masques
for filename in os.listdir(destination_image_dir):
    # Prétraitement des images
    image_path = os.path.join(destination_image_dir, filename)
    image = Image.open(image_path)
    image = image.resize(target_size)
    image = np.array(image, dtype=np.float32) / 255.0  # Normalisation entre 0 et 1
    np.save(os.path.join(preprocessed_image_dir, filename.split('.')[0]), image)

for filename in os.listdir(destination_mask_dir):
    # Prétraitement des masques
    mask_path = os.path.join(destination_mask_dir, filename)
    mask = Image.open(mask_path)
    mask = mask.resize(target_size)
    mask = np.array(mask, dtype=np.uint8)
    np.save(os.path.join(preprocessed_mask_dir, filename.split('.')[0]), mask)

print("Données prétraitées avec succès.")


Données prétraitées avec succès.


In [9]:
# Liste pour stocker les chemins d'images et de masques
image_paths = []
mask_paths = []

# Parcourir les fichiers d'images et vérifier si un masque existe
for filename in os.listdir(preprocessed_image_dir):
    image_path = os.path.join(preprocessed_image_dir, filename)
    mask_filename = filename.split('.')[0] + '.png'
    mask_path = os.path.join(preprocessed_mask_dir, mask_filename)
    
    if os.path.exists(mask_path):
        image_paths.append(image_path)
        mask_paths.append(mask_path)

# Assurez-vous que les listes sont dans le même ordre
image_paths.sort()
mask_paths.sort()

print("Listes d'images et de masques créées avec succès.")

Listes d'images et de masques créées avec succès.


In [12]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation

def atrous_conv_layer(inputs, filters, rate):
    conv = Conv2D(filters, kernel_size=3, dilation_rate=rate, padding='same')(inputs)
    batchnorm = BatchNormalization()(conv)
    activation = Activation('relu')(batchnorm)
    return activation

def build_deeplabv1(input_shape, num_classes):
    inputs = tf.keras.layers.Input(shape=input_shape)

    # Encoder
    conv1 = atrous_conv_layer(inputs, filters=64, rate=1)
    conv2 = atrous_conv_layer(conv1, filters=128, rate=2)
    conv3 = atrous_conv_layer(conv2, filters=256, rate=4)
    
    # Decoder
    # Upsample the feature maps and merge with corresponding encoder features
    upsample1 = tf.keras.layers.UpSampling2D(size=(4, 4))(conv3)
    merge1 = tf.keras.layers.Concatenate()([upsample1, conv2])
    conv4 = atrous_conv_layer(merge1, filters=128, rate=1)
    
    upsample2 = tf.keras.layers.UpSampling2D(size=(4, 4))(conv4)
    merge2 = tf.keras.layers.Concatenate()([upsample2, conv1])
    conv5 = atrous_conv_layer(merge2, filters=64, rate=1)
    
    # Final segmentation layer
    segmentation = Conv2D(num_classes, kernel_size=1, activation='softmax')(conv5)
    
    model = Model(inputs=inputs, outputs=segmentation)
    return model

# Example usage
input_shape = (513, 513, 3)  # Adjust according to your input image dimensions
num_classes = 21  # Number of classes in your dataset
model = build_deeplabv1(input_shape, num_classes)
model.summary()


ValueError: A `Concatenate` layer requires inputs with matching shapes except for the concatenation axis. Received: input_shape=[(None, 2052, 2052, 256), (None, 513, 513, 128)]

In [1]:
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import Input, Conv2D, Conv2DTranspose, BatchNormalization, Activation, Add, UpSampling2D, concatenate
from tensorflow.keras.models import Model

def atrous_spatial_pyramid_pooling(inputs, output_stride):
    """Atrous Spatial Pyramid Pooling (ASPP) module."""
    atrous_rates = [output_stride, 2 * output_stride, 4 * output_stride]
    input_shape = tf.shape(inputs)[1:3]

    # Atrous convolutions with different rates
    conv_1x1_1 = Conv2D(256, (1, 1), padding='same', activation='relu')(inputs)
    conv_3x3_1 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[0], activation='relu')(inputs)
    conv_3x3_2 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[1], activation='relu')(inputs)
    conv_3x3_3 = Conv2D(256, (3, 3), padding='same', dilation_rate=atrous_rates[2], activation='relu')(inputs)

    # Image-level features
    global_avg_pooling = tf.reduce_mean(inputs, (1, 2), name='global_avg_pooling', keepdims=True)
    global_avg_pooling = Conv2D(256, (1, 1), padding='same', activation='relu')(global_avg_pooling)
    global_avg_pooling = UpSampling2D(size=(input_shape[0] // 4, input_shape[1] // 4), interpolation='bilinear')(global_avg_pooling)

    # Concatenate and project the features
    concatenated_features = concatenate([conv_1x1_1, conv_3x3_1, conv_3x3_2, conv_3x3_3, global_avg_pooling], axis=-1)
    concatenated_features = Conv2D(256, (1, 1), padding='same', activation='relu')(concatenated_features)
    concatenated_features = BatchNormalization()(concatenated_features)
    concatenated_features = Activation('relu')(concatenated_features)

    return concatenated_features

def build_deeplab(input_shape, num_classes, output_stride=6):
    # Load pre-trained ResNet50 model
    base_model = ResNet50(include_top=False, weights='imagenet', input_shape=input_shape)
    backbone_output = base_model.output
    
    # Atrous Spatial Pyramid Pooling (ASPP) module
    aspp_output = atrous_spatial_pyramid_pooling(backbone_output, output_stride)
    
    # Upsampling and skip connections
    x = Conv2DTranspose(48, (4, 4), strides=(2, 2), padding='same', activation='relu')(aspp_output)
    skip_connection = base_model.get_layer('conv2_block3_out').output
    x = concatenate([x, skip_connection], axis=-1)
    
    # Final convolution for segmentation
    output = Conv2D(num_classes, (1, 1), padding='same', activation='softmax')(x)
    
    # Create the DeepLab model
    model = Model(inputs=base_model.input, outputs=output)
    
    return model

# Model parameters
# input_shape = (224, 224, 3)  # Input size for the model
input_shape = (513, 513, 3)  # Input size for the model
num_classes = 21  # Number of classes in PASCAL VOC
output_stride = 16  # Output stride for the ASPP module

# Create the DeepLabv3+ model with ResNet backbone
model = build_deeplab(input_shape, num_classes, output_stride)

# Display model summary
model.summary()


Exception: URL fetch failure on https://storage.googleapis.com/tensorflow/keras-applications/resnet/resnet50_weights_tf_dim_ordering_tf_kernels_notop.h5: None -- [WinError 10060] Une tentative de connexion a échoué car le parti connecté n’a pas répondu convenablement au-delà d’une certaine durée ou une connexion établie a échoué car l’hôte de connexion n’a pas répondu

In [ ]:
import tensorflow as tf
from tensorflow.keras.losses import SparseCategoricalCrossentropy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import MeanIoU, Precision, Recall

# Hyperparamètres
batch_size = 8
num_epochs = 50

# Création d'un générateur de données
def data_generator(image_paths, mask_paths, batch_size):
    while True:
        for i in range(0, len(image_paths), batch_size):
            batch_image_paths = image_paths[i : i + batch_size]
            batch_mask_paths = mask_paths[i : i + batch_size]

            batch_images = [np.load(image_path) for image_path in batch_image_paths]
            batch_masks = [np.load(mask_path) for mask_path in batch_mask_paths]

            yield np.array(batch_images), np.array(batch_masks)

# Création du modèle DeepLab
model = build_deeplabv1(input_shape=target_size + (3,), num_classes=num_classes)

# Compilation du modèle
model.compile(optimizer=Adam(learning_rate=1e-4),
              loss=SparseCategoricalCrossentropy(from_logits=True),
              metrics=['accuracy', MeanIoU(num_classes=num_classes), Precision(), Recall()])

# Création des listes d'images et de masques
image_paths = [...]  # Chemins vers les images prétraitées
mask_paths = [...]   # Chemins vers les masques prétraités

# Création des générateurs d'entraînement et de validation
train_generator = data_generator(image_paths[:2000], mask_paths[:2000], batch_size)
val_generator = data_generator(image_paths[2000:], mask_paths[2000:], batch_size)

# Entraînement du modèle
model.fit(train_generator, steps_per_epoch=len(image_paths[:2000]) // batch_size,
          validation_data=val_generator, validation_steps=len(image_paths[2000:]) // batch_size,
          epochs=num_epochs)

print("Entraînement terminé.")

In [ ]:
# Évaluation du modèle sur les données de validation
evaluation_results = model.evaluate(val_generator, steps=len(image_paths[2000:]) // batch_size)

print("Perte de validation:", evaluation_results[0])
print("Précision de validation:", evaluation_results[1])


In [ ]:
from sklearn.model_selection import GridSearchCV

# Définir les hyperparamètres à ajuster
param_grid = {
    'batch_size': [8, 16, 32],
    'learning_rate': [1e-4, 1e-3, 1e-2]
}

# Créer le modèle pour l'entraînement
model = build_deeplabv1(input_shape=target_size + (3,), num_classes=num_classes)

# Créer le générateur de données pour l'entraînement
train_generator = data_generator(image_paths[:2000], mask_paths[:2000], batch_size)

# Utiliser GridSearchCV pour trouver les meilleurs hyperparamètres
grid_search = GridSearchCV(model, param_grid, scoring='accuracy', cv=3)
grid_search.fit(train_generator, steps_per_epoch=len(image_paths[:2000]) // batch_size)

# Afficher les meilleurs hyperparamètres
print("Meilleurs hyperparamètres trouvés:", grid_search.best_params_)


In [ ]:
# Entrer les params trouvés dans le grid search
# Entraînement du modèle
model.fit(train_generator, steps_per_epoch=len(image_paths[:2000]) // batch_size,
          validation_data=val_generator, validation_steps=len(image_paths[2000:]) // batch_size,
          epochs=num_epochs)

In [ ]:
from sklearn.model_selection import KFold

# Définir le nombre de plis pour la validation croisée
num_folds = 5

# Créer un objet KFold
kf = KFold(n_splits=num_folds, shuffle=True)

# Liste pour stocker les scores de validation
validation_scores = []

# Parcourir les plis de la validation croisée
for fold_idx, (train_index, val_index) in enumerate(kf.split(image_paths)):
    print(f"Fold {fold_idx + 1}/{num_folds}")
    
    # Créer le modèle pour l'entraînement
    model = build_deeplabv1(input_shape=target_size + (3,), num_classes=num_classes)
    
    # Créer les générateurs de données pour l'entraînement et la validation
    train_generator = data_generator([image_paths[i] for i in train_index],
                                     [mask_paths[i] for i in train_index], batch_size)
    val_generator = data_generator([image_paths[i] for i in val_index],
                                   [mask_paths[i] for i in val_index], batch_size)
    
    # Entraîner le modèle
    model.fit(train_generator, steps_per_epoch=len(train_index) // batch_size,
              validation_data=val_generator, validation_steps=len(val_index) // batch_size,
              epochs=num_epochs)
    
    # Évaluer le modèle sur les données de validation du pli
    fold_score = model.evaluate(val_generator, steps=len(val_index) // batch_size)
    validation_scores.append(fold_score[1])  # Ajouter la précision à la liste

# Calculer la précision moyenne sur tous les plis
average_precision = sum(validation_scores) / num_folds
print("Précision moyenne sur la validation croisée:", average_precision)

In [ ]:
# Création du générateur de données de test
test_generator = data_generator(image_paths_test, mask_paths_test, batch_size)

# Évaluation du modèle sur les données de test
test_evaluation_results = model.evaluate(test_generator, steps=len(image_paths_test) // batch_size)

print("Perte sur les données de test:", test_evaluation_results[0])
print("Précision sur les données de test:", test_evaluation_results[1])


In [ ]:
# Après l'entraînement du modèle
model.save('chemin/vers/votre/modele.h5')

In [ ]:
import matplotlib.pyplot as plt

# Charger le modèle entraîné (assurez-vous que le chemin est correct)
model = tf.keras.models.load_model('chemin_vers_votre_modele.h5')

# Sélectionner quelques images de validation pour la visualisation
num_images_to_visualize = 5
sample_indices = np.random.choice(len(image_paths[2000:]), num_images_to_visualize, replace=False)

# Préparer les générateurs de données pour la visualisation
visualization_generator = data_generator([image_paths[i] for i in sample_indices + 2000],
                                         [mask_paths[i] for i in sample_indices + 2000], 1)

# Parcourir les images de validation sélectionnées et visualiser les résultats
for i, (image, mask) in enumerate(visualization_generator):
    predicted_mask = model.predict(image)
    predicted_mask = np.argmax(predicted_mask, axis=-1)[0]
    
    plt.figure(figsize=(10, 5))
    
    plt.subplot(1, 3, 1)
    plt.imshow(image[0])
    plt.title("Image originale")
    plt.axis('off')
    
    plt.subplot(1, 3, 2)
    plt.imshow(mask[0], cmap='gray')
    plt.title("Masque réel")
    plt.axis('off')
    
    plt.subplot(1, 3, 3)
    plt.imshow(predicted_mask, cmap='gray')
    plt.title("Masque prédit")
    plt.axis('off')
    
    plt.tight_layout()
    plt.show()

print("Visualisation terminée.")

Utilisation du modèle sur de nouvelles images 

In [ ]:
from skimage import morphology
import cv2


# Charger le modèle entraîné (assurez-vous que le chemin est correct)
model = tf.keras.models.load_model('chemin_vers_votre_modele.h5')

# Chemin vers les images de test
test_image_dir = os.path.join(data_dir, "test_images")

# Dossier de destination pour les résultats de l'inférence
results_dir = os.path.join(data_dir, "inference_results")
os.makedirs(results_dir, exist_ok=True)

# Taille d'entrée du modèle
input_size = (513, 513)

# Parcourir les images de test
for filename in os.listdir(test_image_dir):
    image_path = os.path.join(test_image_dir, filename)
    
    # Charger et prétraiter l'image
    image = cv2.imread(image_path)
    image = cv2.resize(image, input_size)
    image = image / 255.0  # Normalisation
    
    # Effectuer l'inférence
    predicted_mask = model.predict(np.expand_dims(image, axis=0))
    predicted_mask = np.argmax(predicted_mask, axis=-1)[0]
    
    # Appliquer des techniques de post-traitement (par exemple, suppression des petits objets)
    postprocessed_mask = morphology.remove_small_objects(predicted_mask.astype(bool), min_size=64)
    predicted_mask = morphology.dilation(predicted_mask, morphology.disk(3))

    
    # Enregistrer le masque prédit
    mask_image = Image.fromarray((postprocessed_mask * 255).astype(np.uint8))
    mask_image.save(os.path.join(results_dir, filename.split('.')[0] + '_mask.png'))

print("Inférence et post-traitement terminés.")
